# Hotel bookings data generating 

Install Necessary Libraries

Installs pandas for data manipulation, faker for realistic fake data, and numpy for random number generation and handling missing values.

In [11]:
# Install required libraries 
%pip install pandas faker numpy


Note: you may need to restart the kernel to use updated packages.


Import Libraries

Prepares all libraries. Faker creates fake names, dates, etc.; numpy helps with missing values; random for various random choices.

In [12]:
# Import libraries needed for data generation, manipulation, and randomness
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import timedelta, datetime


Initialize Faker and Set Seeds

This ensures your results are repeatable (useful for debugging and re-running) and initializes the Faker object.

In [13]:
#  Set random seeds for reproducibility and initialize Faker
fake = Faker()
random.seed(42)
np.random.seed(42)


Generate Room Data

We create 100 rooms.

Each room has random type, view, capacity, nightly_rate, status, and floor.

Exports data to "Rooms.csv".

In [14]:
# Generate Room data for the "Rooms" table
room_types = ["Single", "Double", "Suite", "Deluxe", "Family"]
views = ["Sea", "Garden", "City", "Mountain"]
statuses = ["available", "occupied", "maintenance"]

num_rooms = 100      # Fewer rooms than guests/bookings (realistic in hotels)
rooms = []

for room_id in range(1, num_rooms + 1):
    room = {
        "room_id": room_id,
        "room_type": random.choice(room_types),
        "view": random.choice(views),
        "capacity": random.randint(1, 5),
        "nightly_rate": round(random.uniform(50, 500), 2),
        "status": random.choices(statuses, weights=[0.7, 0.25, 0.05])[0],
        "floor": random.randint(1, 10)
    }
    rooms.append(room)

rooms_df = pd.DataFrame(rooms)
rooms_df.to_csv("Rooms.csv", index=False)
rooms_df.head()


,room_id,room_type,view,capacity,nightly_rate,status,floor
0,1,Single,Sea,3,160.20,available,2
1,2,Family,Sea,5,239.86,available,4
2,3,Double,Sea,5,139.48,available,9
3,4,Deluxe,Garden,4,315.17,occupied,1
4,5,Double,Mountain,3,175.04,available,6


Generate Guest Data

400 guests, each with random name, gender, age, signup date.

Deliberate missing values for phone_number (~5%) and loyalty_status (~5%).

Exports data to "Guests.csv".

In [15]:
# Generate Guest data for the "Guests" table (with some missing values)
loyalty_statuses = ["Bronze", "Silver", "Gold", None]
gender_choices = ["Male", "Female", "Other"]

num_guests = 400      # 400 guests for this scenario
guests = []

for guest_id in range(1, num_guests + 1):
    guest = {
        "guest_id": guest_id,
        "full_name": fake.name(),
        "gender": random.choice(gender_choices),
        "age": random.randint(18, 80),
        "phone_number": fake.phone_number() if random.random() > 0.05 else None,  # ~5% missing
        "loyalty_status": random.choices(loyalty_statuses, weights=[0.5, 0.3, 0.15, 0.05])[0],  # ~5% missing
        "signup_date": fake.date_between(start_date="-5y", end_date="today").strftime("%Y-%m-%d"),
    }
    guests.append(guest)

guests_df = pd.DataFrame(guests)
guests_df.to_csv("Guests.csv", index=False)
guests_df.head()


,guest_id,full_name,gender,age,phone_number,loyalty_status,signup_date
0,1,James Franco,Other,56,908.641.8618x7941,Silver,2021-08-20
1,2,Eric Beasley,Male,79,+1-903-416-6092x4865,Silver,2021-02-19
2,3,Dennis Rogers,Female,54,+1-444-775-3976,None,2023-12-30
3,4,Stacy Young,Female,78,497.595.5210x0677,Bronze,2021-01-16
4,5,Joanna Butler,Other,71,001-213-716-8322x839,Bronze,2020-12-10


Generate Booking Data

1200 bookings: each links to a valid guest and room, with random stay duration and calculated total price.

payment_status is mostly "Completed".

rating is missing in roughly 40% for realism.

All foreign keys are valid due to the way guest_id and room_id are generated.

Exports data to "Bookings.csv".

In [16]:
# Generate Booking data for the "Bookings" table (with some missing values)
payment_status_choices = ["Pending", "Completed", "Cancelled"]
num_bookings = 1200     # 1200 bookings; more than number of guests/rooms

bookings = []

for booking_id in range(1, num_bookings + 1):
    guest_id = random.randint(1, num_guests)
    room_id = random.randint(1, num_rooms)
    checkin_dt = fake.date_between(start_date='-2y', end_date='today')
    stay_length = random.randint(1, 14)
    checkout_dt = checkin_dt + timedelta(days=stay_length)
    nightly_rate = float(rooms_df.loc[rooms_df['room_id'] == room_id, "nightly_rate"].values[0])
    total_price = round(nightly_rate * stay_length, 2)

    booking = {
        "booking_id": booking_id,
        "guest_id": guest_id,
        "room_id": room_id,
        "checkin_date": checkin_dt.strftime("%Y-%m-%d"),
        "checkout_date": checkout_dt.strftime("%Y-%m-%d"),
        "total_price": total_price,
        "payment_status": random.choices(payment_status_choices, weights=[0.1, 0.85, 0.05])[0],
        "rating": random.randint(1, 5) if random.random() > 0.4 else None,   # 40% missing
    }
    bookings.append(booking)

bookings_df = pd.DataFrame(bookings)
bookings_df.to_csv("Bookings.csv", index=False)
bookings_df.head()


,booking_id,guest_id,room_id,checkin_date,checkout_date,total_price,payment_status,rating
0,1,227,84,2025-06-05,2025-06-10,1253.30,Completed,NaN
1,2,276,35,2024-03-04,2024-03-13,3399.30,Completed,NaN
2,3,316,95,2024-10-20,2024-10-30,1317.80,Completed,NaN
3,4,272,29,2025-02-11,2025-02-22,1777.16,Completed,NaN
4,5,170,92,2025-06-29,2025-07-07,1382.56,Completed,1.0


Check for Duplicates/Foreign Key Consistency

Sanity checks. Code will raise an error if there are any uniqueness problems or FK mismatches. This is for peace of mind.

In [ ]:
# Confirm there are no duplicate PKs or foreign key errors
assert guests_df['guest_id'].is_unique, "Duplicate Guest IDs!"
assert rooms_df['room_id'].is_unique, "Duplicate Room IDs!"
assert bookings_df['booking_id'].is_unique, "Duplicate Booking IDs!"

# Check for valid foreign key references ONLY (all guest_ids and room_ids in bookings must exist in corresponding tables)
assert set(bookings_df['guest_id']).issubset(set(guests_df['guest_id'])), "Booking references non-existent guest!"
assert set(bookings_df['room_id']).issubset(set(rooms_df['room_id'])), "Booking references non-existent room!"
print("All keys are valid and unique.")


All keys are valid and unique.


In [18]:
#  Preview a few records from each table
print("Sample guests:")
print(guests_df.head(), "\n")
print("Sample rooms:")
print(rooms_df.head(), "\n")
print("Sample bookings:")
print(bookings_df.head())


Sample guests:
   guest_id      full_name  gender  age          phone_number loyalty_status  \
0         1   James Franco   Other   56     908.641.8618x7941         Silver   
1         2   Eric Beasley    Male   79  +1-903-416-6092x4865         Silver   
2         3  Dennis Rogers  Female   54       +1-444-775-3976           None   
3         4    Stacy Young  Female   78     497.595.5210x0677         Bronze   
4         5  Joanna Butler   Other   71  001-213-716-8322x839         Bronze   

  signup_date  
0  2021-08-20  
1  2021-02-19  
2  2023-12-30  
3  2021-01-16  
4  2020-12-10   

Sample rooms:
   room_id room_type      view  capacity  nightly_rate     status  floor
0        1    Single       Sea         3        160.20  available      2
1        2    Family       Sea         5        239.86  available      4
2        3    Double       Sea         5        139.48  available      9
3        4    Deluxe    Garden         4        315.17   occupied      1
4        5    Double  Mount

In [19]:
#  Count missing values in each column of each table
print("Missing values in Guests table:")
print(guests_df.isnull().sum())

print("\nMissing values in Rooms table:")
print(rooms_df.isnull().sum())

print("\nMissing values in Bookings table:")
print(bookings_df.isnull().sum())


Missing values in Guests table:
guest_id           0
full_name          0
gender             0
age                0
phone_number      11
loyalty_status    19
signup_date        0
dtype: int64

Missing values in Rooms table:
room_id         0
room_type       0
view            0
capacity        0
nightly_rate    0
status          0
floor           0
dtype: int64

Missing values in Bookings table:
booking_id          0
guest_id            0
room_id             0
checkin_date        0
checkout_date       0
total_price         0
payment_status      0
rating            478
dtype: int64
